# Phase 3 — End-to-end nonconvex $\ell_p$-Lasso benchmark

The benchmark augments the isolated scalar-prox experiments with **objective vs iteration**, **objective vs online time**, **independent float64 reference-FB residual**, **support-location comparisons**, and **common-tolerance time-to-stop**. It does *not* measure a global optimum.

- **A–C:** repository setup, tests, short CPU check.
- **D:** optional full A100 run (explicit opt-in).
- **E:** inspect results, including `all_hit` flags.
- **F:** export the complete raw-data ZIP.

Keep raw result archives outside the public Git repository. No reviewer report or response material is included in this notebook.

## A. Clone the public source branch and install

In [ ]:
from pathlib import Path
import subprocess, os
ROOT = Path('/content/Fast_Lp_Prox')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','revision/major-experiments',
                    'https://github.com/Jy-U1/Fast_Lp_Prox.git',str(ROOT)],check=True)
else:
    # Reuse the clone but refresh the public experimental branch.
    subprocess.run(['git','-C',str(ROOT),'fetch','origin','revision/major-experiments'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','revision/major-experiments'],check=True)
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','revision/major-experiments'],check=True)
os.chdir(ROOT)
print('Repository:', ROOT)
subprocess.run(['git','rev-parse','--short','HEAD'],check=True)
subprocess.run(['python','-m','pip','install','-e','.[dev]'],check=True)

## B. Verify the new experiment code

In [ ]:
subprocess.run(['python','-m','pytest','-q','tests/test_phase3_lasso.py'],check=True)

## C. Small CPU smoke test (does not establish GPU speed)

In [ ]:
from pathlib import Path
SMOKE = ROOT / 'results/phase3/lasso_smoke_colab'
RUN_SMOKE = True
if RUN_SMOKE:
    if not (SMOKE / 'metadata.json').exists():
        subprocess.run(['python','revision/phase3/scripts/run_lasso_end_to_end.py',
        '--device','cpu','--m','60','--n','100','--support','10','--seeds','0',
        '--max-iter','60','--verify-every','10','--tol','1e-3',
        '--n-fit','512','--n-val','1001','--repeats','2','--warmup','0',
        '--checked-stop-repeats','1',
        '--output-dir',str(SMOKE)],check=True)
    subprocess.run(['python','revision/phase3/scripts/rebuild_lasso_artifacts.py',
                    '--root',str(SMOKE)],check=True)
    print((SMOKE/'SUMMARY.md').read_text()[:2800])

## D. Full A100 experiment — opt in explicitly

Choose a **GPU runtime** in Colab before running. The default compares Chebyshev $D=6$, Newton $K=3$, and Bisection $K=20$ on **three separately seeded** instances. The main $p=1/3$ workload follows the dimensions and data model used in the paper. This uses an independent high-precision reference at checkpoints and can take substantially longer than the scalar benchmark.

Set `RUN_FORMAL=True` after the tests pass. A second $p=3/4$ workload is optional and should be run under a new output directory. Do not invent success when a method fails to reach the common stopping tolerance.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
RUN_FORMAL = False  # Change to True after CPU smoke succeeds.
FORMAL = ROOT / 'results/phase3/lasso_formal_p0333'
if RUN_FORMAL:
    assert torch.cuda.is_available(), 'Select GPU runtime first'
    if not (FORMAL / 'metadata.json').exists():
        subprocess.run(['python','revision/phase3/scripts/run_lasso_end_to_end.py',
        '--device','cuda','--dtype','float32','--p','1/3','--p-label','1/3',
        '--m','2000','--n','4000','--support','200','--sigma','0.01',
        '--lambda-reg','0.05','--seeds','0,1,2',
        '--max-iter','500','--verify-every','10','--tol','1e-4',
        '--power-iterations','100','--ref-bisect-iters','80',
        '--candidates','chebyshev:6,newton:3,bisection:20',
        '--n-fit','8192','--n-val','80001',
        '--repeats','5','--warmup','2','--checked-stop-repeats','2',
        '--output-dir',str(FORMAL)],check=True)
    subprocess.run(['python','revision/phase3/scripts/rebuild_lasso_artifacts.py',
                    '--root',str(FORMAL)],check=True)

## E. Inspect convergence, runtime and support before reporting

In [ ]:
import pandas as pd
from IPython.display import display, Image
SOURCE = FORMAL if FORMAL.joinpath('run_summary.csv').exists() else SMOKE
print('Inspecting:', SOURCE)
summary = pd.read_csv(SOURCE/'run_summary.csv')
summary_agg = pd.read_csv(SOURCE/'across_seed_summary.csv')
display(summary_agg)
display(summary[['seed','method','verified_stop_hit',
                 'first_verified_stop_iteration','replay_budget_median_ms',
                 'checked_stop_median_wall_ms','final_objective',
                 'final_verified_displacement','support_jaccard_vs_reference_FB',
                 'support_exact_vs_reference_FB']])
if not summary_agg['all_hit'].all():
    print('IMPORTANT: Not every method reached the common tolerance. Do not claim a common time-to-target across all methods yet.')
for fn in sorted((SOURCE/'figures').glob('objective_vs_iteration_seed*.png'))[:1]:
    display(Image(filename=str(fn)))
for fn in sorted((SOURCE/'figures').glob('reference_displacement_seed*.png'))[:1]:
    display(Image(filename=str(fn)))
for fn in sorted((SOURCE/'figures').glob('objective_vs_online_time_seed*.png'))[:1]:
    display(Image(filename=str(fn)))

## F. Export data and figures as a separate archive

In [ ]:
import zipfile, datetime
from google.colab import files
ROOT_RESULTS = FORMAL if FORMAL.joinpath('run_summary.csv').exists() else SMOKE
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%SZ')
kind = 'formal' if ROOT_RESULTS == FORMAL else 'smoke'
archive = ROOT / f'FastLp_Phase3_Lasso_{kind}_{stamp}.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ROOT_RESULTS.rglob('*')):
        if path.is_file():
            z.write(path,arcname='phase3/results/'+path.relative_to(ROOT_RESULTS).as_posix())
    for source in [ROOT/'revision/phase3/README.md',
                   ROOT/'revision/phase3/scripts/run_lasso_end_to_end.py',
                   ROOT/'revision/phase3/scripts/rebuild_lasso_artifacts.py']:
        z.write(source,arcname='phase3/source/'+source.name)
print('Exported:',archive,'size(MB):',round(archive.stat().st_size/1e6,3))
files.download(str(archive))